# 04 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific **Clean** and **Noisy** champions from  (generated by Notebook 03).
2. Executes each champion **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Evaluates Clean Champions on clean settings () and Noisy Champions on noisy settings ().
4. Attaches IOH Analyzer context manager to output IOH performance files to .


In [ ]:
import sys
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path(".").resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection
from synthesis.execution import AlgorithmExecutor

# ── User Execution Controls & Selective Filters ──────────────────────────────
FORCE_REEVALUATE  = False   # Set True to bypass cache and re-evaluate all
FILTER_MODELS     = None    # e.g., ["14b"], ["7b"], or None for all models
FILTER_PROBLEMS   = None    # e.g., [8, 11] to evaluate specific problems, or None for all
FILTER_STRATEGIES = None    # e.g., ["baseline", "guided"] or None for all
FILTER_MODES      = None    # e.g., ["clean"], ["noisy"], or None for all
FILTER_DIMS       = None    # e.g., [2, 3], or None (uses all DIMS in database)
N_RUNS            = 10      # Number of independent benchmark runs per config
TIMEOUT_SECONDS   = 30.0    # Per-run execution timeout in seconds

# ── Dynamic Experiment Parameters Extracted Purely from Database ──────────────
CHAMPIONS_PATH = DATA_DIR / "champions.json"
EVALUATIONS_DIR   = RESULTS_DIR / "evaluations"

with get_db_connection() as conn:
    df_exp_meta = pd.read_sql_query(
        "SELECT DISTINCT dim, budget, noise_std, problem_id FROM experiments WHERE status = 'completed'",
        conn
    )

if df_exp_meta.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

DIMS   = sorted(df_exp_meta["dim"].dropna().astype(int).unique().tolist())
BUDGET = int(df_exp_meta["budget"].dropna().max())

# Apply FILTER_DIMS if specified
if FILTER_DIMS is not None:
    DIMS = [d for d in DIMS if d in FILTER_DIMS]

print(f"🎯 Dynamic parameters loaded from database:")
print(f"  • Evaluated Dimensions: {DIMS}")
print(f"  • Benchmark Budget:     {BUDGET}")
print(f"  • Runs per config:      {N_RUNS}")
print(f"  • Champions JSON:       {CHAMPIONS_PATH}")
print(f"  • Evaluations Output:      {EVALUATIONS_DIR}")
print(f"  • Force Re-evaluate:    {FORCE_REEVALUATE}")
if FILTER_MODELS:     print(f"  • Filter Models:        {FILTER_MODELS}")
if FILTER_PROBLEMS:   print(f"  • Filter Problems:      f{FILTER_PROBLEMS}")
if FILTER_STRATEGIES: print(f"  • Filter Strategies:    {FILTER_STRATEGIES}")
if FILTER_MODES:      print(f"  • Filter Modes:         {FILTER_MODES}")


## 1. Load Champions JSON

In [ ]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 03 first.')

with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions_raw = json.load(f)

# Support both nested {model: {key: info}} and legacy flat {key: info}
champions_flat = {}
for k, v in champions_raw.items():
    if isinstance(v, dict) and 'code_path' in v:
        champions_flat[k] = v
    elif isinstance(v, dict):
        for sub_k, sub_v in v.items():
            champions_flat[f'{k}/{sub_k}'] = sub_v

print(f'Loaded {len(champions_flat)} champion configuration(s) across {len(champions_raw)} model category(ies):')
for model_key, model_dict in champions_raw.items():
    if isinstance(model_dict, dict) and 'code_path' not in model_dict:
        print(f'  • {model_key}: {len(model_dict)} champions')


import shutil
import hashlib
import re
import ioh
from collections import defaultdict

executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

def _compute_code_hash(code_str: str) -> str:
    """Return SHA256 hex digest of the raw algorithm code."""
    return hashlib.sha256(code_str.strip().encode("utf-8")).hexdigest()

def _read_provenance(prov_path: Path) -> dict | None:
    """Read provenance metadata JSON if present."""
    if prov_path.exists():
        try:
            return json.loads(prov_path.read_text(encoding="utf-8"))
        except Exception:
            return None
    return None

def _write_provenance(prov_path: Path, info: dict, dim: int, noise_std: float, code_hash: str, med_err: float):
    """Write evaluation provenance to target IOH log directory."""
    prov = {
        "algorithm_name":    info["algorithm_name"],
        "experiment_id":     int(info.get("experiment_id", -1)),
        "iteration_id":      int(info.get("iteration_id", -1)),
        "code_path":         info["code_path"],
        "code_hash":         code_hash,
        "dim":               dim,
        "noise_std":          noise_std,
        "problem_id":         int(info["problem_id"]),
        "prompt_strategy":   info.get("prompt_strategy", "baseline"),
        "llm_name":          info.get("llm_name", ""),
        "mode":              info.get("mode", "all"),
        "n_runs":            N_RUNS,
        "median_clean_error": float(med_err) if not np.isinf(med_err) else None,
        "evaluated_at":      pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding="utf-8")

def get_model_slug(llm_name: str) -> str:
    """Extract clean, concise model slug from DB model name (e.g. qwen_14b, qwen_7b)."""
    l = str(llm_name).lower()
    if '14b' in l:
        return 'qwen_14b'
    elif '7b' in l:
        return 'qwen_7b'
    elif '70b' in l:
        return 'qwen_70b'
    else:
        m = re.search(r'([a-zA-Z0-9]+).*?(\d+b)', l)
        if m:
            return f"{m.group(1)}_{m.group(2)}"
        return l.removesuffix('.gguf').replace('-', '_').replace('.', '_').split('/')[-1]

evaluated = []
skipped = []
current_model_header = None

print("=== Starting Evaluation of Champions ===")

for key, info in champions_flat.items():
    p_id       = int(info["problem_id"])
    dim        = int(info["dim"])
    mode       = info.get("mode", "all").lower()
    strat      = info.get("prompt_strategy", "baseline").lower()
    llm_name   = info.get("llm_name", key.split("/")[0])
    noise_std  = float(info.get("noise_std", 0.0))
    exp_id     = int(info.get("experiment_id", -1))
    algo_name  = info["algorithm_name"]
    clean_key  = key.split("/")[-1]

    # User Filters
    if FILTER_MODELS and not any(m.lower() in llm_name.lower() for m in FILTER_MODELS): continue
    if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS: continue
    if FILTER_STRATEGIES and strat not in FILTER_STRATEGIES: continue
    if FILTER_MODES and mode not in FILTER_MODES: continue
    if FILTER_DIMS and dim not in FILTER_DIMS: continue

    # Print distinct model section header when switching between LLM models
    if llm_name != current_model_header:
        current_model_header = llm_name
        print()
        print("="*75)
        print(f"📦 Evaluating Champions for LLM Model: [{llm_name}]")
        print("="*75)

    code_file = PROJECT_ROOT / info["code_path"] if not Path(info["code_path"]).is_absolute() else Path(info["code_path"])
    if not code_file.exists():
        print(f"[WARN] Code file for {clean_key} not found at {code_file}. Skipping.")
        continue

    code_content = code_file.read_text(encoding="utf-8")
    code_hash    = _compute_code_hash(code_content)

    # Destination path in results/evaluations (Option 1: model-prefixed solver folder)
    out_dir = EVALUATIONS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
    model_slug = get_model_slug(llm_name)
    folder_name = f"{model_slug}_{strat}_{mode}"
    target_log_folder = out_dir / folder_name
    prov_path = target_log_folder / "provenance.json"

    # Provenance Check (cache validation)
    if not FORCE_REEVALUATE and target_log_folder.exists():
        prov = _read_provenance(prov_path)
        dat_files = [f for f in target_log_folder.glob("**/*.dat") if f.stat().st_size > 0]
        if prov and prov.get("code_hash") == code_hash and len(dat_files) > 0:
            skipped.append((llm_name, clean_key))
            continue

    print(f"⚡ Evaluating [{llm_name}] {clean_key} ({algo_name}, Exp #{exp_id}) for {N_RUNS} runs...")
    # Delete existing target log folder if re-evaluating to prevent duplicate -1, -2 suffixes
    if target_log_folder.exists():
        shutil.rmtree(target_log_folder, ignore_errors=True)
    target_log_folder.mkdir(parents=True, exist_ok=True)

    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    run_errors = []

    # Initialize logger ONCE for all N_RUNS of this condition
    logger = ioh.logger.Analyzer(
        root=str(out_dir),
        folder_name=folder_name,
        algorithm_name=f"LLaMEA-{llm_name}/{strat}",
        store_positions=False
    )

    for run_idx in range(1, N_RUNS + 1):
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=run_idx,
            noise_strategy=noise_strat,
        )
        problem.attach_logger(logger)

        try:
            x_opt, f_opt = executor.execute_algorithm(
                code=code_content,
                name=algo_name,
                dim=dim,
                problem=problem,
                budget=BUDGET
            )
            clean_prob = BBOBProblem(problem_id=p_id, dim=dim, instance_id=run_idx, noise_strategy=NoNoiseStrategy())
            final_err = clean_prob(x_opt) if x_opt is not None else float(f_opt)
            run_errors.append(final_err)
        except Exception as e:
            print(f"   ❌ Run {run_idx}/{N_RUNS} failed: {e}")
            run_errors.append(float("inf"))
        finally:
            if hasattr(problem, "clean_problem") and hasattr(problem.clean_problem, "detach_logger"):
                problem.clean_problem.detach_logger()

    del logger
    med_err = np.median(run_errors) if run_errors else float("inf")
    _write_provenance(prov_path, info, dim, noise_std, code_hash, med_err)
    evaluated.append((llm_name, clean_key))
    print(f"   ✅ Completed {len(run_errors)}/{N_RUNS} runs (Median Error: {med_err:.4e})")

print()
print("="*75)
print(f"🎯 Evaluation Summary: {len(evaluated)} evaluated, {len(skipped)} skipped (cached)")


In [ ]:
import shutil
import hashlib
import re
import ioh
from collections import defaultdict

executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

def _compute_code_hash(code_str: str) -> str:
    """Return SHA256 hex digest of the raw algorithm code."""
    return hashlib.sha256(code_str.strip().encode("utf-8")).hexdigest()

def _read_provenance(prov_path: Path) -> dict | None:
    """Read provenance metadata JSON if present."""
    if prov_path.exists():
        try:
            return json.loads(prov_path.read_text(encoding="utf-8"))
        except Exception:
            return None
    return None

def _write_provenance(prov_path: Path, info: dict, dim: int, noise_std: float, code_hash: str, med_err: float):
    """Write evaluation provenance to target IOH log directory."""
    prov = {
        "algorithm_name":    info["algorithm_name"],
        "experiment_id":     int(info.get("experiment_id", -1)),
        "iteration_id":      int(info.get("iteration_id", -1)),
        "code_path":         info["code_path"],
        "code_hash":         code_hash,
        "dim":               dim,
        "noise_std":          noise_std,
        "problem_id":         int(info["problem_id"]),
        "prompt_strategy":   info.get("prompt_strategy", "baseline"),
        "llm_name":          info.get("llm_name", ""),
        "mode":              info.get("mode", "all"),
        "n_runs":            N_RUNS,
        "median_clean_error": float(med_err) if not np.isinf(med_err) else None,
        "evaluated_at":      pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding="utf-8")

def get_model_slug(llm_name: str) -> str:
    """Extract clean, concise model slug from DB model name (e.g. qwen_14b, qwen_7b)."""
    l = str(llm_name).lower()
    if "14b" in l:
        return "qwen_14b"
    elif "7b" in l:
        return "qwen_7b"
    elif "70b" in l:
        return "qwen_70b"
    else:
        m = re.search(r"([a-zA-Z0-9]+).*?(\d+b)", l)
        if m:
            return f"{m.group(1)}_{m.group(2)}"
        return l.removesuffix(".gguf").replace("-", "_").replace(".", "_").split("/")[-1]

evaluated = []
skipped = []
current_model_header = None

print("=== Starting Evaluation of Champions ===")

for key, info in champions_flat.items():
    p_id       = int(info["problem_id"])
    dim        = int(info["dim"])
    mode       = info.get("mode", "all").lower()
    strat      = info.get("prompt_strategy", "baseline").lower()
    llm_name   = info.get("llm_name", key.split("/")[0])
    noise_std  = float(info.get("noise_std", 0.0))
    exp_id     = int(info.get("experiment_id", -1))
    algo_name  = info["algorithm_name"]
    clean_key  = key.split("/")[-1]

    # User Filters
    if FILTER_MODELS and not any(m.lower() in llm_name.lower() for m in FILTER_MODELS): continue
    if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS: continue
    if FILTER_STRATEGIES and strat not in FILTER_STRATEGIES: continue
    if FILTER_MODES and mode not in FILTER_MODES: continue
    if FILTER_DIMS and dim not in FILTER_DIMS: continue

    # Print distinct model section header when switching between LLM models
    if llm_name != current_model_header:
        current_model_header = llm_name
        print(f"📦 Evaluating Champions for LLM Model: [{llm_name}]")
        print("="*75)

    code_file = PROJECT_ROOT / info["code_path"] if not Path(info["code_path"]).is_absolute() else Path(info["code_path"])
    if not code_file.exists():
        print(f"[WARN] Code file for {clean_key} not found at {code_file}. Skipping.")
        continue

    code_content = code_file.read_text(encoding="utf-8")
    code_hash    = _compute_code_hash(code_content)

    # Destination path in results/evaluations (Option 1: model-prefixed solver folder)
    out_dir = EVALUATIONS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
    model_slug = get_model_slug(llm_name)
    folder_name = f"{model_slug}_{strat}"
    target_log_folder = out_dir / folder_name
    prov_path = target_log_folder / "provenance.json"

    # Provenance Check (cache validation)
    if not FORCE_REEVALUATE and target_log_folder.exists():
        prov = _read_provenance(prov_path)
        dat_files = [f for f in target_log_folder.glob("**/*.dat") if f.stat().st_size > 0]
        if prov and prov.get("code_hash") == code_hash and len(dat_files) > 0:
            skipped.append((llm_name, clean_key))
            continue

    print(f"⚡ Evaluating [{llm_name}] {clean_key} ({algo_name}, Exp #{exp_id}) for {N_RUNS} runs...")
    # Delete existing target log folder if re-evaluating to prevent duplicate -1, -2 suffixes
    if target_log_folder.exists():
        shutil.rmtree(target_log_folder, ignore_errors=True)
    target_log_folder.mkdir(parents=True, exist_ok=True)

    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    run_errors = []

    # Initialize logger ONCE for all N_RUNS of this condition
    logger = ioh.logger.Analyzer(
        root=str(out_dir),
        folder_name=folder_name,
        algorithm_name=f"LLaMEA-{llm_name}/{strat}",
        store_positions=False
    )

    for run_idx in range(1, N_RUNS + 1):
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=run_idx,
            noise_strategy=noise_strat,
        )
        problem.attach_logger(logger)

        try:
            x_opt, f_opt = executor.execute_algorithm(
                code=code_content,
                name=algo_name,
                dim=dim,
                problem=problem,
                budget=BUDGET
            )
            clean_prob = BBOBProblem(problem_id=p_id, dim=dim, instance_id=run_idx, noise_strategy=NoNoiseStrategy())
            final_err = clean_prob(x_opt) if x_opt is not None else float(f_opt)
            run_errors.append(final_err)
        except Exception as e:
            print(f"   ❌ Run {run_idx}/{N_RUNS} failed: {e}")
            run_errors.append(float("inf"))
        finally:
            if hasattr(problem, "clean_problem") and hasattr(problem.clean_problem, "detach_logger"):
                problem.clean_problem.detach_logger()

    del logger
    med_err = np.median(run_errors) if run_errors else float("inf")
    _write_provenance(prov_path, info, dim, noise_std, code_hash, med_err)
    evaluated.append((llm_name, clean_key))
    print(f"   ✅ Completed {len(run_errors)}/{N_RUNS} runs (Median Error: {med_err:.4e})")

print(f"🎯 Evaluation Summary: {len(evaluated)} evaluated, {len(skipped)} skipped (cached)")
